# Satellite Data Analysis for Jamaica
## Notebook 2 of 4. Turn Satellite Pictures into Hectares

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

> **STUDENT EDITION.** Cells marked **YOUR TURN** have gaps to fill in. Look
> for `____` and `# TODO`. Many gaps list three options in a comment; one is
> right and the others teach you something by being wrong. Everything else runs
> as given. If you get stuck, read the hint under the cell before asking.
> The day: two hours of missions, then the one-hour Satellite Challenge.

---

# Mission 2: Measure Portmore's Water in Hectares 🌿

**Your question.** How much of a place is living green, how much is water, and
how much is bare concrete, measured in hectares?

**Why it matters.** After a drought or a flood, nobody acts on "it looks bad".
They act on hectares, and on where. Farmers, water agencies and town planners all
speak the same unit, and this notebook teaches you to hand it to them.

**Your objective.** Measure the open water inside Portmore, one of the most
flood-watched towns in Jamaica. First to the number wins.

You have the pictures from Mission 1. Now you turn them into evidence.

### What you will be able to do by the end

1. Turn two bands into one meaningful number, using a single formula
2. Map plants, water and concrete, and convert each to hectares
3. Report an answer a farmer, planner or relief agency can act on

**Time in class:** about 25 minutes.
**Before you start:** Notebook 1, working.

### Words for this notebook

| Word | What it means here |
|---|---|
| index | A single number computed from two bands that means something physical |
| composite | One clean image stacked from many cloudy ones |
| median | The middle value of a sorted list; ignores freak values |
| mask | A true/false map of which pixels to keep |
| hectare | 100 m by 100 m, roughly a football pitch |
| time series | The same measurement repeated through time |

---

## Part 1. Setting up

Same two cells as last time.

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

---

## Part 2. One formula: the normalized difference

Nearly every satellite index has the same shape:

$$\text{index} = \frac{A - B}{A + B}$$

Pick two bands. Subtract. Divide by the sum. The division cancels out
brightness, so a shaded hillside scores the same as a sunny one. The answer
always lands between -1 and +1.

### YOUR TURN 1

Write the formula yourself. The `+ 1e-10` at the end stops the code crashing if
both bands happen to be zero, which happens in the deepest shadows.

In [ ]:
def my_index(a, b):
    """Normalized difference between two bands."""
    # TODO: return the normalized difference of a and b.
    # Options:  (a - b) / (a + b + 1e-10)   /   (a + b) / (a - b)   /   abs(a - b)
    return ____

# Tests. All four must pass.
assert abs(my_index(0.4, 0.1) - 0.6) < 1e-6,  "Check the order: a minus b on top."
assert abs(my_index(0.1, 0.4) + 0.6) < 1e-6,  "Swapping the inputs should flip the sign."
assert abs(my_index(0.3, 0.3) - 0.0) < 1e-6,  "Equal bands should give zero."
assert abs(my_index(0.8, 0.0) - 1.0) < 1e-6,  "Maximum is 1."
print("All tests pass. That function is now the backbone of everything below.")

*Hint: `(a - b) / (a + b + 1e-10)`*

### The indices you will actually use

| Name | Formula | What it finds | Why those two bands |
|---|---|---|---|
| **NDVI** | (NIR - Red) / (NIR + Red) | Living plants | Leaves eat red light, bounce infrared |
| **NDWI** | (Green - NIR) / (Green + NIR) | Open water | Water reflects green, swallows infrared |
| **MNDWI** | (Green - SWIR) / (Green + SWIR) | Water, sharper edges | Better at telling wet sand from sea |
| **NDBI** | (SWIR - NIR) / (SWIR + NIR) | Concrete, roofs, roads | Built surfaces are dry and bright in SWIR |
| **NBR** | (NIR - SWIR) / (NIR + SWIR) | Burnt ground | Fire kills the infrared signal |

The names are initials. NDVI is the Normalized Difference Vegetation Index, NDWI the Normalized Difference Water Index, MNDWI the Modified NDWI, NDBI the Normalized Difference Built-up Index, and NBR the Normalized Burn Ratio. MNDWI is the one to reach for near water, and Notebook 3 leans on it hard.

---

## Part 3. NDVI: map plant health over Kingston

NDVI reads like a scoreboard:

| NDVI | What is down there |
|---|---|
| below 0 | Water |
| 0 to 0.1 | Concrete, asphalt, bare rock, roofs |
| 0.1 to 0.2 | Bare soil, dry ground |
| 0.2 to 0.4 | Sparse grass, scrub, stressed crops |
| 0.4 to 0.6 | Healthy grass, growing crops |
| above 0.6 | Dense forest, mature canopy, cane at full height |

Rules of thumb, not laws. A cane field can score 0.75 in October and 0.45 in
March with nothing wrong.

In [ ]:
kingston = PLACES["kingston"]
grid = make_grid(kingston, metres=20)

found = search_scenes(kingston, "2025-01-01", "2025-06-30", max_cloud=20)
scene = best_scene(found, grid, min_covered=0.95, min_clear=0.90)
date = scene["properties"]["datetime"][:10]
print("Using", scene["id"], "from", date)

red = read_reflectance(scene, "red", grid)
nir = read_reflectance(scene, "nir", grid)

In [ ]:
# TODO: compute NDVI.
# Options for the two slots: nir / red / green. The table above settles the order.
ndvi = normalized_difference(____, ____)

print(f"NDVI range: {np.nanmin(ndvi):.2f} to {np.nanmax(ndvi):.2f}")
print(f"Average:    {np.nanmean(ndvi):.3f}")

assert np.nanmean(ndvi) > 0, "Negative average means you have the bands backwards."
show(ndvi, f"NDVI over Kingston, {date}", cmap="RdYlGn", vmin=-0.3, vmax=0.9, bar=True)

*Hint: plants reflect infrared strongly, so the infrared band is the larger one.
The larger band goes first.*

### From a coloured map to hectares

Slice the NDVI values at the thresholds above and count the pixels in each
slice. A picture persuades nobody; a number in hectares does.

In [ ]:
# TODO: fill in the missing threshold values from the table above.
# Options for the three blanks: 0.0 / 0.4 / 0.6, each used exactly once.
classes = {
    "Water":            ndvi < ____,
    "Built or bare":   (ndvi >= 0.0)  & (ndvi < 0.2),
    "Sparse growth":   (ndvi >= 0.2)  & (ndvi < ____),
    "Healthy growth":  (ndvi >= 0.4)  & (ndvi < 0.6),
    "Dense canopy":     ndvi >= ____,
}

total = area_hectares(np.isfinite(ndvi), grid)
print(f"Study area: {total:,.0f} hectares\n")
print(f"{'Class':18s} {'Hectares':>10s} {'Share':>8s}")
print("-" * 38)
for name, mask in classes.items():
    ha = area_hectares(mask, grid)
    print(f"{name:18s} {ha:10,.0f} {100 * ha / total:7.1f}%")

*Hint: read the NDVI table again. Water is below 0. Sparse growth stops where
healthy growth starts. Dense canopy starts at 0.6.*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                               gridspec_kw={"width_ratios": [1.1, 1]})

colour_map = {"Water": "#1a5c8a", "Built or bare": "#b0b0b0", "Sparse growth": "#d9c26a",
              "Healthy growth": "#6ab04c", "Dense canopy": "#186a3b"}
painted = np.zeros(ndvi.shape + (3,))
for i, (name, mask) in enumerate(classes.items()):
    rgb_hex = colour_map[name].lstrip("#")
    painted[mask] = [int(rgb_hex[j:j+2], 16) / 255 for j in (0, 2, 4)]

ax1.imshow(painted); ax1.set_title(f"Kingston land cover, {date}")
ax1.set_xticks([]); ax1.set_yticks([]); ax1.grid(False)

names = list(classes)
areas = [area_hectares(classes[n], grid) for n in names]
ax2.barh(names, areas, color=[colour_map[n] for n in names])
ax2.set_xlabel("Hectares"); ax2.set_title("How much of each")
ax2.invert_yaxis()
for i, v in enumerate(areas):
    ax2.text(v, i, f" {v:,.0f}", va="center", fontsize=9)

plt.tight_layout(); plt.show()

---

## Part 4. MNDWI: map the water, then the concrete

Same formula, different bands. MNDWI finds water, NDBI finds concrete.
Notebook 3 leans on MNDWI to trace a coastline.

In [ ]:
green  = read_reflectance(scene, "green", grid)
swir   = read_reflectance(scene, "swir16", grid)

# TODO: build the two remaining indices using the table in Part 2.
# Options for all four slots: green / nir / swir. The table settles which goes where.
mndwi = normalized_difference(____, ____)   # water
ndbi  = normalized_difference(____, ____)   # built-up

water = mndwi > 0.0
built = (ndbi > 0.0) & (~water)

print(f"Water:    {area_hectares(water, grid):8,.0f} ha")
print(f"Built up: {area_hectares(built, grid):8,.0f} ha")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img, ttl, cm in [(axes[0], ndvi, "NDVI: plants", "RdYlGn"),
                          (axes[1], mndwi, "MNDWI: water", "Blues"),
                          (axes[2], ndbi, "NDBI: built ground", "pink")]:
    ax.imshow(img, cmap=cm, vmin=-0.6, vmax=0.6); ax.set_title(ttl)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

*Hint: for MNDWI, water is bright in green and dark in shortwave infrared, so
green goes first. For NDBI, concrete is bright in shortwave infrared, so that
band goes first.*

---

## 📍 Your spot: how green?

Same spot as Notebook 1. One number tells you how alive it is.

In [ ]:
MY_LAT, MY_LON = ____, ____            # your spot from Notebook 1

spot = [MY_LON - 0.02, MY_LAT - 0.02, MY_LON + 0.02, MY_LAT + 0.02]
sg = make_grid(spot, metres=10)
sc = best_scene(search_scenes(spot, "2025-01-01", "2025-12-31", max_cloud=20),
                sg, min_covered=0.95, min_clear=0.7)

score = float(np.nanmean(normalized_difference(
    read_reflectance(sc, "nir", sg), read_reflectance(sc, "red", sg))))

print(f"Your spot scores NDVI {score:.2f}")
print("0.6+ lush | 0.4+ growing | 0.2+ dry or sparse | under 0.2 built or bare | negative water")

---

## 🏁 Mission objective: how much water sits inside Portmore?

Portmore is home to almost 200,000 people and sits low against Hunts Bay and
wetland, which makes it one of the most flood-watched towns in Jamaica. Any flood
plan starts from a baseline: how much open water is normally inside the box,
before a storm surge pushes it into the streets.

**Find the value.** Hectares of water in the Portmore box, from the best scene of
early 2025.

Complete the blank, run it, and call out your number. First correct answer wins.

In [ ]:
portmore = [-76.92, 17.93, -76.86, 17.99]
pg = make_grid(portmore, metres=20)

sc = best_scene(search_scenes(portmore, "2025-01-01", "2025-06-30", max_cloud=20),
                pg, min_covered=0.95, min_clear=0.8)

mnd = normalized_difference(read_reflectance(sc, "green", pg),
                            read_reflectance(sc, "swir16", pg))

# One blank: water is where MNDWI is above what? Options: 0.0 / 0.5 / -0.5
water_ha = area_hectares(mnd > ____, pg)
print(f"THE VALUE: {water_ha:,.0f} hectares of water")

*Stuck? Think back to what the water index does. Water shows up where MNDWI is
positive and land where it is negative, so the boundary you want is the line
between the two. And 0.5 would only catch the deepest, cleanest water, missing
most of the bay.*

---

## Mission 2 complete

- One formula, `(A - B) / (A + B)`, generates every index in common use
- Dividing by the sum removes brightness and leaves surface properties
- NDVI, MNDWI and NDBI find plants, water and concrete
- Thresholds turn a picture into hectares, which is what people act on
- Cloud must be masked, not tolerated, and the SCL band does the labelling
- A median stack of many cloudy scenes gives one clean picture
- A time series over three years shows the seasons directly
- **The archive returns 200 results and never says so. Paginate.**

Next notebook: Hurricane Melissa. You measure the damage across Black River from
orbit, in hectares, and then measure whether Negril's beach moved.

---

### Before you close this notebook

Save a copy to your own Drive (`File` then `Save a copy in Drive`). The next
notebook assumes you have this one working.

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2 from
the European Space Agency, hosted by Amazon; NASA POWER climate records. No API
keys, no fees, no permission needed.*